# edasynth: the main results in the browser

This notebook repeats the main results of [github.com/cbrg0/edasynth](https://github.com/cbrg0/edasynth) without installing anything on your computer. To run it, choose *Runtime* in the menu and then *Run all*. It has three parts.

- **Part 1** recomputes the map of the 300 scenarios and the main figure from the table of all runs stored in the repository, and prints the results of every check. It takes less than a minute.
- **Part 2** downloads the public recordings and repeats the comparison on real data. It takes about ten minutes.
- **Part 3** runs the automatic tests of the program. It takes about two minutes.

The times are approximate, because they depend on the speed of the computer and of the internet connection. The full computation of the 300 scenarios, each repeated 50 times, takes one to two days on a laptop and is therefore not repeated here. Its 11 300 results are stored in `result-tables/individual-runs/grid-metrics-utd.csv`, and Part 1 draws its conclusions from that file.

In [ ]:
# Preparation, which takes about a minute.
# In Colab this cell downloads the repository and installs the program.
# In Jupyter on your own computer, open the notebook inside the repository folder;
# then nothing is downloaded.
import os, subprocess, sys
if os.path.exists("../program-code/edasynth"):
    os.chdir("..")
elif not os.path.exists("program-code/edasynth"):
    subprocess.run(["git", "clone", "-q", "https://github.com/cbrg0/edasynth.git"], check=True)
    os.chdir("edasynth")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]"], check=True)
sys.path.insert(0, "program-code")  # lets Python find the program without restarting the notebook
print("The notebook works in the folder", os.getcwd())

## Part 1. The map of scenarios, recomputed from the table of all runs

The gain Δ is the AUPRC of a detector that sees the EDA and acceleration features minus the AUPRC of a detector that sees only the EDA features. AUPRC is a score from 0 to 1 that shows how well a detector ranks real artifacts above clean parts of the signal. The letter *c* stands for the share of artifacts caused by movement. FB denotes the four coupling features, which check whether the EDA changes right after a movement, and FA4 denotes the four features of movement strength. A scenario counts as useful when the mean gain is at least 0.01 and the lower bound of its 95 % interval is above zero.

In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

from edasynth import cli
from edasynth.config import loadConfig
from edasynth.experiment import summarise
from edasynth.figures import f0Summary

cfg = loadConfig(None)
grid = cli.readGrid(Path("result-tables/individual-runs/grid-metrics-utd.csv"), cfg)  # the 11 300 computed runs; cells that are equivalent are filled in by copying
print(f"The table covers {grid.groupby(['c','kappa','deltaMax','piR','classes']).ngroups} scenarios with {len(grid)} runs; 11 300 of them were computed, and the rest are copies for scenarios that give the same data.")

by = ["c", "kappa", "deltaMax", "piR", "classes"]
useful = {d: int(summarise(grid, by, d, cfg["analysis"]["decision"])["useful"].sum()) for d in ("dA", "dA4", "dB")}
print("Number of useful scenarios out of 300 for each feature set (dA: 96 features, dA4: 4 movement-strength features, dB: 4 coupling features):", useful)

mix = grid[grid.classes == "C1+C2+C3+C5"]
print("\nMean gain in the mixture of all artifact classes for each value of c:")
print(mix.groupby("c")[["dA", "dA4", "dB"]].mean().round(3))

In [ ]:
summary = pd.read_csv("result-tables/grid-summary-utd.csv")
f0Summary(summary, Path("colab-figures"), cfg["analysis"]["observedDelta"])
display(Image("colab-figures/F0-summary.png", width=640))

### The checks of the model and the checks on independent data

G3 and G4 test whether a classifier can tell synthetic windows from real ones. An AUC of 0.5 would mean that it cannot tell them apart at all, and the limit fixed in advance is 0.70. EDABE is a data set from another device, labelled by other experts, and WESAD provides the movements of another wristband.

In [ ]:
v = json.load(open("result-tables/validation-utd.json"))
print("G2, the model measures known physiology again:", "PASS" if v["G2"]["passed"] else "FAIL")
print(f"G3, real against synthetic clean windows: median AUC {v['G3']['auc']:.3f}, the limit is 0.70")
print(f"G4, real against synthetic artifact windows: median AUC {v['G4']['auc']:.3f}, the limit is 0.70")
print("G5, no gain without a link:", "PASS" if v["G5"]["passed"] else "no gain, but small losses; this was accepted as an exception")
print("G6, a link built into the data is found:", "PASS" if v["G6"]["passed"] else "FAIL")

e = json.load(open("result-tables/external-edabe.json"))
print(f"\nEDABE, another device and other experts: real against synthetic artifact windows AUC {e['E1z']['median']:.3f}, clean windows AUC {e['E2z']['median']:.3f}, the limit is 0.70")
print(f"On EDABE, a detector trained on synthetic data reaches AUPRC {e['E3a']['median']:.3f}, and a detector trained on the real UTD recordings reaches {e['E3b']:.3f}")

w = json.load(open("result-tables/external-wesad.json"))
for cell in w["cells"]:
    print(f"With WESAD movements at c = {cell['c']}, the coupling features gain {cell['dB_wesad']['mean']:+.3f}; with UTD movements they gain {cell['dB_utd']['mean']:+.3f}")

## Part 2. The result on real data, which takes about ten minutes

This part downloads the PhysioNet recordings and the expert labels of Zhang et al. (2017) and checks that the files are exactly the expected ones by their SHA-256 checksums. It then compares a detector that sees only the EDA features with a detector that also sees the acceleration. Each detector is trained on 19 participants and tested on the remaining one, in turn for every participant. For the logistic regression the expected result is an AUPRC of 0.4486 without acceleration and 0.4754 with it, a gain of +0.0269, with a 95 % interval from −0.0022 to +0.0580.

In [ ]:
!bash data-download-and-external-checks/download-data.sh
!{sys.executable} -m edasynth realdata --stored

## Part 3. The automatic tests of the program, which take about two minutes

In [ ]:
!{sys.executable} -m pytest -q -m "not slow"